In [1]:
! pip install mysql-connector-python

In [2]:
import pandas as pd 
import numpy as np
import mysql.connector 

In [3]:
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='shubham',
    database='py_project'
)

In [4]:
conn.is_connected()

True

In [5]:
#we will create an function out of it
def connect_to_db():
    return mysql.connector.connect(
    host='localhost',
    user='root',
    password='shubham',
    database='py_project'
)

In [6]:
connect_to_db().is_connected()

True

In [7]:
db=connect_to_db()
cursor=db.cursor(dictionary=True)

In [8]:
cursor

In [9]:
"select count(*) as total_suppliers from suppliers"

'select count(*) as total_suppliers from suppliers'

In [10]:
cursor.execute("select count(*) as total_suppliers from suppliers")
row = cursor.fetchone()
list(row.values())[0]

50

In [11]:
queries = {
"Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

"Total Products": "SELECT COUNT(*) AS count FROM products",

"Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

"Total Sale Value (Last 3 Months)": """
SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
FROM stock_entries se
JOIN products p ON se.product_id = p.product_id
WHERE se.change_type = 'Sale'
AND se.entry_date >= (
SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
""",

"Total Restock Value (Last 3 Months)": """
SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
FROM stock_entries se
JOIN products p ON se.product_id = p.product_id
WHERE se.change_type = 'Restock'
AND se.entry_date >= (
SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
""",

"Below Reorder & No Pending Reorders": """
SELECT COUNT(*) AS below_reorder
FROM products p
WHERE p.stock_quantity < p.reorder_level
AND p.product_id NOT IN (
SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
"""
}

In [12]:
result={}
for label, query in queries.items():
    cursor.execute(query)
    row= cursor.fetchone()
    result[label]=list(row.values())[0]

In [13]:
result


{'Total Suppliers': 50,
 'Total Products': 200,
 'Total Categories Dealing': 5,
 'Total Sale Value (Last 3 Months)': 1529482.99,
 'Total Restock Value (Last 3 Months)': 6691663.1,
 'Below Reorder & No Pending Reorders': 14}

In [14]:
## Now lets create function out of it 

def get_basic_info(cursor):
    """
    Retrieve summary inventory and supply chain metrics.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.

    Returns:
        dict: Dictionary of metric labels and their values.
    """

    queries = {
        "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

        "Total Products": "SELECT COUNT(*) AS count FROM products",

        "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

        "Total Sale Value (Last 3 Months)": """
            SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Sale'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Total Restock Value (Last 3 Months)": """
            SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Restock'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Below Reorder & No Pending Reorders": """
            SELECT COUNT(*) AS below_reorder
            FROM products p
            WHERE p.stock_quantity < p.reorder_level
              AND p.product_id NOT IN (
                  SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
        """
    }

    results = {}
    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        # Since row is a dictionary, extract the single value by getting the first value in dict.values()
        results[label] = list(row.values())[0]

In [15]:
result

{'Total Suppliers': 50,
 'Total Products': 200,
 'Total Categories Dealing': 5,
 'Total Sale Value (Last 3 Months)': 1529482.99,
 'Total Restock Value (Last 3 Months)': 6691663.1,
 'Below Reorder & No Pending Reorders': 14}

In [19]:
queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

tables = {}
for label, query in queries.items():
    cursor.execute(query)
    tables[label] = cursor.fetchall()



In [20]:
queries

{'Suppliers Contact Details': 'SELECT supplier_name, contact_name, email, phone FROM suppliers',
 'Products with Supplier and Stock': '\n            SELECT \n                p.product_name,\n                s.supplier_name,\n                p.stock_quantity,\n                p.reorder_level\n            FROM products p\n            JOIN suppliers s ON p.supplier_id = s.supplier_id\n            ORDER BY p.product_name ASC\n        ',
 'Products Needing Reorder': '\n            SELECT product_name, stock_quantity, reorder_level\n            FROM products\n            WHERE stock_quantity <= reorder_level\n        '}

In [22]:
def add_new_manual_id(cursor, db, p_name , p_category , p_price , p_stock , p_reorder, p_supplier):
    proc_call= "call AddNewProductManualID(%s, %s, %s ,%s ,%s, %s)"
    params= (p_name , p_category , p_price , p_stock , p_reorder, p_supplier)
    cursor.execute(proc_call, params)
    db.commit()

In [ ]:
def get_categories(cursor):
    cursor.execute("select Distinct category  from products  order by category  asc")
    rows= cursor.fetchall()

In [24]:
get_categories(cursor)

In [25]:
def get_suppliers(cursor):
    cursor.execute("select supplier_id , supplier_name from suppliers order by  supplier_name asc")
    return cursor.fetchall()

In [26]:
suppliers= get_suppliers(cursor)

In [27]:
supplier_ids=[s["supplier_id"] for s in suppliers]
supplier_names=[s["supplier_name"] for s in suppliers]

In [28]:
supplier_names

['Anderson-Thompson',
 'Armstrong-Vance',
 'Barker, White and Carson',
 'Barrett Ltd',
 'Baxter-Meadows',
 'Charles Inc',
 'Clark Group',
 'Douglas Ltd',
 'Elliott-Ayers',
 'Evans Inc',
 'Franklin, Kane and Price',
 'Freeman-Gordon',
 'Gallagher-Miller',
 'Gomez PLC',
 'Hall-Brown',
 'Harris-Cummings',
 'Henderson LLC',
 'Hensley-Branch',
 'Hudson Inc',
 'Johnson-Bass',
 'Kaufman Ltd',
 'Lawrence, Garcia and Hernandez',
 'Lloyd and Sons',
 'Mann-Marshall',
 'Mendoza-Jones',
 'Middleton LLC',
 'Miller-Martinez',
 'Moody-Vang',
 'Morgan Inc',
 'Morgan-Andrews',
 'Moss-Evans',
 'Newton, Valencia and Carr',
 'Ortega-Mahoney',
 'Patrick, Walter and Harrison',
 'Perez, Price and Wallace',
 'Reynolds-Phillips',
 'Rogers-Greene',
 'Rowe PLC',
 'Rowland Ltd',
 'Smith, Kennedy and Moreno',
 'Stewart, Williams and Cox',
 'Taylor-Love',
 'Tran LLC',
 'Tucker-Arnold',
 'Turner-Davis',
 'Vega, Cook and Miller',
 'Williams Ltd',
 'Wilson, Graham and Williams',
 'Wong Group',
 'Young, Browning and War